# 43 — LSEG/Gemma story-family first-release robustness

**Question.** Does preventing later revisions of the same LSEG story family
from receiving separate weight improve Notebook 39's frozen continuous Gemma
construction?

**Why this is new.** The 888,155-headline scorer population deduplicates exact
normalized text, but distinct headline revisions can still have different hashes.
LSEG's terminal numeric `story_id` suffix provides a return-independent family
definition that has not been used in Notebooks 11–42.

**Frozen comparison before loading prices.** Keep only headline hashes that are
the earliest timestamped release of at least one story family. Everything else
remains Notebook 39's rule: original 33 firms, mean continuous Gemma score,
q90−q10 dispersion at or above its strictly-prior trailing-60 q75 after 40
observations, two names per leg, 25% cap, gross one, dollar neutral, h1, and
10 bps per traded side. The filter proceeds to returns only if it removes at
least 500 hashes, retains the 167-session rectangular input panel, leaves at
least 15 active dates, and changes at least one active date or selected basket.

**Evidence status.** The LSEG return window has already been opened repeatedly.
This is one bounded retrospective mechanism comparison, not confirmation. The
declared two-test family is filtered-minus-baseline and filtered-minus-cash;
both receive BH correction. No parameter may be tuned from the result.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "final_experiments":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from final_experiments.lib.aggregators import build_aggregation_panel  # noqa: E402
from final_experiments.lib.evaluate import annualized_sharpe  # noqa: E402
from final_experiments.lib.lseg_expanded import (  # noqa: E402
    load_price_exports,
    load_success_scores,
    map_headlines_to_entry_sessions,
)
from final_experiments.lib.lseg_story_families import (  # noqa: E402
    load_first_release_hashes,
)
from final_experiments.lib.plots import CATEGORICAL, INK, apply_house_style  # noqa: E402
from final_experiments.lib.risk_overlay import (  # noqa: E402
    paired_block_bootstrap_difference,
)
from final_experiments.lib.sector_portfolios import SECTOR_MEMBERS  # noqa: E402
from final_experiments.lib.sparse_spread import ledger_rows_to_frame  # noqa: E402
from sentiment_benchmark.artifact_io import sha256_file  # noqa: E402
from sentiment_benchmark.strategy_research.ledger import (  # noqa: E402
    run_open_to_open_ledger,
)
from sentiment_benchmark.strategy_research.market import OpenToOpenReturn  # noqa: E402
from sentiment_benchmark.strategy_research.portfolio import (  # noqa: E402
    PositionTarget,
    TargetPortfolio,
)
from sentiment_benchmark.trading_effectiveness import (  # noqa: E402
    benjamini_hochberg,
)

SEED = 20260816
COST_BPS = 10.0
COST_GRID = (0.0, 1.0, 2.0, 5.0, 10.0)
BLOCK_LENGTH = 5
BOOTSTRAP_REPLICATIONS = 9_999
TRAILING_SESSIONS = 60
MIN_TRAILING_SESSIONS = 40
EVENT_INTENSITY_QUANTILE = 0.75
TOP_BOTTOM_NAMES = 2
SINGLE_NAME_CAP = 0.25
INITIAL_NAV = 1_000_000.0
MIN_REMOVED_HASHES = 500
MIN_ACTIVE_DATES = 15

COLLECTION_DIR = REPO_ROOT / "Data/collections/lseg_us_sector_44_8m_headlines/derived"
GEMMA_PATH = (
    COLLECTION_DIR
    / "headline_scores_gemma4_26b_a4b_it_deepinfra_fp8_investor_headline_soft_label_v1.csv"
)
MERGED_HEADLINES = COLLECTION_DIR / "merged/headlines.jsonl"
SECTOR33_PRICE = REPO_ROOT / "Data/derived/prices/lseg_us_sector_33_8m.csv"
ADD11_PRICE = REPO_ROOT / "Data/derived/prices/lseg_us_sector_add11_8m.csv"
NOTEBOOK39_MANIFEST = (
    REPO_ROOT / "final_experiments/outputs/39_gemma_continuous_portability/manifest.json"
)
OUTPUT_DIR = (
    REPO_ROOT
    / "final_experiments/outputs/43_lseg_gemma_story_family_first_release"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ORIGINAL_SYMBOLS = tuple(
    symbol for members in SECTOR_MEMBERS.values() for symbol in members[:3]
)
SYMBOLS = tuple(sorted(ORIGINAL_SYMBOLS))
apply_house_style()
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

## Bind Notebook 39 and build the licence-safe family filter

This stage reads score metadata and raw story identifiers locally. The helper
returns hashes and counts only; no headline text is written to notebook output.
No price value or return has been loaded yet.

In [2]:
notebook39 = json.loads(NOTEBOOK39_MANIFEST.read_text(encoding="utf-8"))
expected_spec = {
    "firm_open_reducer": "mean_continuous",
    "long_names": 2,
    "short_names": 2,
    "single_name_cap": 0.25,
    "gross_exposure": 1.0,
    "holding_intervals": 1,
    "cost_bps_per_side_for_future_test": 10.0,
    "current_return_window_evaluated": False,
}
for key, expected in expected_spec.items():
    if notebook39["selected_future_specification"].get(key) != expected:
        raise ValueError(f"Notebook 39 specification mismatch for {key}")
if notebook39["adaptive_schedule"] != {
    "top_bottom_names": 2,
    "trailing_sessions": 60,
    "minimum_trailing_sessions": 40,
    "event_intensity_quantile": 0.75,
}:
    raise ValueError("Notebook 39 adaptive schedule changed")

gemma_scores, score_audit = load_success_scores(GEMMA_PATH, scorer="gemma4_26b")
first_release_hashes, family_audit = load_first_release_hashes(
    MERGED_HEADLINES,
    expected_hashes=set(gemma_scores["headline_sha256"]),
)
filtered_scores = gemma_scores.loc[
    gemma_scores["headline_sha256"].isin(first_release_hashes)
].copy()
if len(filtered_scores) != family_audit["retained_first_release_hashes"]:
    raise RuntimeError("first-release score count does not reconcile")

display(
    pd.Series(
        {
            "complete successful hashes": len(gemma_scores),
            "story families": family_audit["story_families_in_score_population"],
            "multirow families": family_audit["families_with_multiple_rows"],
            "retained first-release hashes": len(filtered_scores),
            "removed later-revision-only hashes": family_audit[
                "removed_later_revision_only_hashes"
            ],
            "removed share": family_audit["removed_share"],
        },
        name="input-only story-family audit",
    ).to_frame()
)

,input-only story-family audit
complete successful hashes,"888,155.000000"
story families,"1,149,954.000000"
multirow families,"3,638.000000"
retained first-release hashes,"879,841.000000"
removed later-revision-only hashes,"8,314.000000"
removed share,0.009361


## Return-free firm-open and schedule comparison

The two price exports are read with `symbol` and `session_date` only. The
baseline must reproduce Notebook 39's 5,511-row panel and 40 active dates.

In [3]:
calendar_parts = []
for path in (SECTOR33_PRICE, ADD11_PRICE):
    calendar_parts.append(pd.read_csv(path, usecols=["symbol", "session_date"]))
calendar = pd.concat(calendar_parts, ignore_index=True)
calendar["symbol"] = calendar["symbol"].astype(str)
calendar["session_date"] = pd.to_datetime(calendar["session_date"]).dt.normalize()
if calendar.duplicated(["symbol", "session_date"]).any():
    raise ValueError("LSEG session calendar has duplicate symbol-session rows")


def aggregate_input(scores: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    events = map_headlines_to_entry_sessions(scores, calendar)
    events = events.loc[events["symbol"].isin(ORIGINAL_SYMBOLS)].copy()
    keys = (
        events[["symbol", "entry_session"]]
        .drop_duplicates()
        .rename(columns={"entry_session": "session_date"})
    )
    keys["split"] = "retrospective_lseg"
    panel = build_aggregation_panel(keys, stories=events)
    panel = panel.sort_values(["session_date", "symbol"], kind="mergesort").reset_index(
        drop=True
    )
    return panel, len(events)


baseline_panel, baseline_associations = aggregate_input(gemma_scores)
filtered_panel, filtered_associations = aggregate_input(filtered_scores)
expected_summary = notebook39["full_26b_input_summary"]
if baseline_associations != expected_summary["original33_associations"]:
    raise RuntimeError("baseline association count does not reproduce Notebook 39")
if len(baseline_panel) != expected_summary["firm_open_rows"]:
    raise RuntimeError("baseline firm-open count does not reproduce Notebook 39")


def build_schedule(panel: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    counts = panel.groupby("session_date")["symbol"].nunique()
    complete = counts.loc[counts.eq(len(SYMBOLS))].index
    use = panel.loc[panel["session_date"].isin(complete)].copy()
    sessions = pd.DatetimeIndex(sorted(use["session_date"].unique()))
    spread = use.groupby("session_date")["mean_continuous"].apply(
        lambda values: float(values.quantile(0.90) - values.quantile(0.10))
    )
    spread = spread.reindex(sessions)
    threshold = spread.shift(1).rolling(
        TRAILING_SESSIONS, min_periods=MIN_TRAILING_SESSIONS
    ).quantile(EVENT_INTENSITY_QUANTILE)
    active = spread.ge(threshold) & threshold.notna()
    weights = np.zeros((len(sessions), len(SYMBOLS)), dtype=float)
    rows = []
    for row_index, session in enumerate(sessions):
        day = (
            use.loc[use["session_date"].eq(session), ["symbol", "mean_continuous"]]
            .set_index("symbol")
            .reindex(SYMBOLS)
        )
        if day["mean_continuous"].isna().any():
            raise RuntimeError("complete input session contains a missing mean score")
        ordered = day.reset_index().sort_values(
            ["mean_continuous", "symbol"], kind="mergesort"
        )
        short_symbols = tuple(ordered.head(TOP_BOTTOM_NAMES)["symbol"])
        long_symbols = tuple(ordered.tail(TOP_BOTTOM_NAMES)["symbol"])
        if set(long_symbols) & set(short_symbols):
            raise RuntimeError("continuous rank legs overlap")
        if bool(active.loc[session]):
            long_indices = [SYMBOLS.index(symbol) for symbol in long_symbols]
            short_indices = [SYMBOLS.index(symbol) for symbol in short_symbols]
            weights[row_index, long_indices] = 0.5 / TOP_BOTTOM_NAMES
            weights[row_index, short_indices] = -0.5 / TOP_BOTTOM_NAMES
        rows.append(
            {
                "session_date": session,
                "dispersion": float(spread.loc[session]),
                "trailing_threshold": (
                    float(threshold.loc[session])
                    if pd.notna(threshold.loc[session])
                    else float("nan")
                ),
                "active": bool(active.loc[session]),
                "long_symbols": "|".join(sorted(long_symbols))
                if bool(active.loc[session])
                else "",
                "short_symbols": "|".join(sorted(short_symbols))
                if bool(active.loc[session])
                else "",
            }
        )
    if np.abs(weights).max() > SINGLE_NAME_CAP + 1e-12:
        raise RuntimeError("frozen target violates the single-name cap")
    if not np.allclose(weights.sum(axis=1), 0.0, atol=1e-12):
        raise RuntimeError("frozen targets are not dollar neutral")
    return pd.DataFrame(rows), weights


baseline_schedule, baseline_weights = build_schedule(baseline_panel)
filtered_schedule, filtered_weights = build_schedule(filtered_panel)
if len(baseline_schedule) != 167 or int(baseline_schedule["active"].sum()) != 40:
    raise RuntimeError("baseline adaptive schedule does not reproduce Notebook 39")
if not baseline_schedule["session_date"].equals(filtered_schedule["session_date"]):
    raise RuntimeError("first-release filter changes the complete session clock")

panel_pair = baseline_panel[["session_date", "symbol", "mean_continuous"]].merge(
    filtered_panel[["session_date", "symbol", "mean_continuous"]],
    on=["session_date", "symbol"],
    suffixes=("_baseline", "_first_release"),
    validate="one_to_one",
)
panel_pair["changed"] = ~np.isclose(
    panel_pair["mean_continuous_baseline"],
    panel_pair["mean_continuous_first_release"],
    rtol=0.0,
    atol=1e-12,
)
schedule_pair = baseline_schedule.merge(
    filtered_schedule,
    on="session_date",
    suffixes=("_baseline", "_first_release"),
    validate="one_to_one",
)
schedule_pair["decision_changed"] = (
    schedule_pair["active_baseline"].ne(schedule_pair["active_first_release"])
    | schedule_pair["long_symbols_baseline"].ne(
        schedule_pair["long_symbols_first_release"]
    )
    | schedule_pair["short_symbols_baseline"].ne(
        schedule_pair["short_symbols_first_release"]
    )
)
baseline_active = set(
    schedule_pair.loc[schedule_pair["active_baseline"], "session_date"]
)
filtered_active = set(
    schedule_pair.loc[schedule_pair["active_first_release"], "session_date"]
)
active_union = baseline_active | filtered_active
active_jaccard = (
    len(baseline_active & filtered_active) / len(active_union)
    if active_union
    else float("nan")
)
daily_rank_spearman = panel_pair.groupby("session_date").apply(
    lambda day: day["mean_continuous_baseline"].corr(
        day["mean_continuous_first_release"], method="spearman"
    ),
    include_groups=False,
)
input_comparison = pd.DataFrame(
    [
        {
            "baseline_associations": baseline_associations,
            "first_release_associations": filtered_associations,
            "baseline_firm_opens": len(baseline_panel),
            "first_release_firm_opens": len(filtered_panel),
            "changed_firm_opens": int(panel_pair["changed"].sum()),
            "median_daily_rank_spearman": float(daily_rank_spearman.median()),
            "baseline_active_dates": len(baseline_active),
            "first_release_active_dates": len(filtered_active),
            "active_date_jaccard": active_jaccard,
            "changed_decision_dates": int(schedule_pair["decision_changed"].sum()),
        }
    ]
)
input_gate = bool(
    family_audit["removed_later_revision_only_hashes"] >= MIN_REMOVED_HASHES
    and len(filtered_schedule) == 167
    and len(filtered_active) >= MIN_ACTIVE_DATES
    and int(schedule_pair["decision_changed"].sum()) >= 1
)
display(input_comparison.T)
print("Return-free feasibility gate:", input_gate)
if not input_gate:
    raise RuntimeError("first-release screen stopped at the return-free feasibility gate")

,0
baseline_associations,"800,145.000000"
first_release_associations,"791,037.000000"
baseline_firm_opens,"5,511.000000"
first_release_firm_opens,"5,511.000000"
changed_firm_opens,"1,938.000000"
median_daily_rank_spearman,0.998663
baseline_active_dates,40.000000
first_release_active_dates,43.000000
active_date_jaccard,0.930233
changed_decision_dates,8.000000


Return-free feasibility gate: True


## Open the already-used LSEG return window once for this frozen comparison

Split-adjusted open-to-open returns, exact drifted-weight turnover, terminal
liquidation, and the 10-bps convention are unchanged. Every selected position
must have a finite next-open return; otherwise the notebook fails closed.

In [4]:
prices = load_price_exports(SECTOR33_PRICE, ADD11_PRICE, expected_symbols=44)
prices = prices.loc[prices["symbol"].isin(ORIGINAL_SYMBOLS)].copy()
prices = prices.sort_values(["symbol", "session_date"], kind="mergesort")
prices["next_session"] = prices.groupby("symbol", sort=False)["session_date"].shift(-1)
prices["raw_open_h1"] = (
    prices.groupby("symbol", sort=False)["adjusted_open"].shift(-1)
    / prices["adjusted_open"]
    - 1.0
)
sessions = pd.DatetimeIndex(baseline_schedule["session_date"])
return_matrix = (
    prices.pivot(index="session_date", columns="symbol", values="raw_open_h1")
    .reindex(index=sessions, columns=SYMBOLS)
    .to_numpy(dtype=float)
)
next_session_frame = (
    prices.loc[prices["session_date"].isin(sessions), ["session_date", "next_session"]]
    .drop_duplicates()
    .sort_values("session_date", kind="mergesort")
)
if next_session_frame["session_date"].duplicated().any() or len(next_session_frame) != len(sessions):
    raise RuntimeError("LSEG next-session mapping is not common across the 33 firms")
next_sessions = pd.DatetimeIndex(next_session_frame["next_session"])
selected_union = (~np.isclose(baseline_weights, 0.0)) | (
    ~np.isclose(filtered_weights, 0.0)
)
missing_selected = selected_union & ~np.isfinite(return_matrix)
outcome_audit = pd.DataFrame(
    [
        {
            "selected_position_cells": int(selected_union.sum()),
            "missing_selected_returns": int(missing_selected.sum()),
            "affected_sessions": int(missing_selected.any(axis=1).sum()),
        }
    ]
)
display(outcome_audit.T)
if missing_selected.any():
    raise RuntimeError("a frozen selected LSEG position lacks a next-open return")
safe_returns = np.where(np.isfinite(return_matrix), return_matrix, 0.0)


def matrix_backtest(
    weights: np.ndarray,
    returns: np.ndarray,
    *,
    cost_bps: float,
) -> pd.DataFrame:
    if weights.shape != returns.shape or weights.shape[0] != len(sessions):
        raise ValueError("matrix backtest inputs do not align")
    current = np.zeros(weights.shape[1], dtype=float)
    rows = []
    rate = cost_bps / 10_000.0
    for session, next_session, target, asset_returns in zip(
        sessions, next_sessions, weights, returns, strict=True
    ):
        turnover = float(np.abs(target - current).sum())
        cost = rate * turnover
        gross = float(np.dot(target, asset_returns))
        net = gross - cost
        if 1.0 + net <= 0:
            raise RuntimeError("strategy NAV became non-positive")
        current = target * (1.0 + asset_returns) / (1.0 + net)
        rows.append(
            {
                "session_date": session,
                "next_session": next_session,
                "gross_return": gross,
                "net_return": net,
                "turnover": turnover,
                "cost": cost,
                "gross_exposure": float(np.abs(target).sum()),
                "active_names": int(np.count_nonzero(target)),
            }
        )
    liquidation_turnover = float(np.abs(current).sum())
    liquidation_cost = rate * liquidation_turnover
    rows[-1]["net_return"] = (
        (1.0 + rows[-1]["net_return"]) * (1.0 - liquidation_cost) - 1.0
    )
    rows[-1]["turnover"] += liquidation_turnover
    rows[-1]["cost"] += liquidation_cost
    return pd.DataFrame(rows)


def summarize(daily: pd.DataFrame, *, cost_bps: float) -> dict[str, float | int]:
    gross = daily["gross_return"].to_numpy(dtype=float)
    net = daily["net_return"].to_numpy(dtype=float)
    equity = np.cumprod(1.0 + net)
    with_start = np.r_[1.0, equity]
    total_turnover = float(daily["turnover"].sum())
    return {
        "n_sessions": len(daily),
        "active_sessions": int(daily["gross_exposure"].gt(0).sum()),
        "mean_gross": float(gross.mean()),
        "mean_net": float(net.mean()),
        "sharpe_gross": annualized_sharpe(pd.Series(gross)),
        "sharpe_net": annualized_sharpe(pd.Series(net)),
        "total_return_gross": float(np.prod(1.0 + gross) - 1.0),
        "total_return_net": float(equity[-1] - 1.0),
        "max_drawdown": float(
            np.min(with_start / np.maximum.accumulate(with_start) - 1.0)
        ),
        "mean_turnover": float(total_turnover / len(daily)),
        "total_turnover": total_turnover,
        "breakeven_bps_per_side": (
            float(10_000.0 * gross.sum() / total_turnover)
            if total_turnover > 0
            else float("nan")
        ),
        "cost_bps_per_side": cost_bps,
    }


daily_paths = {
    "all_headlines": matrix_backtest(
        baseline_weights, safe_returns, cost_bps=COST_BPS
    ),
    "first_release_only": matrix_backtest(
        filtered_weights, safe_returns, cost_bps=COST_BPS
    ),
}
strategy_results = pd.DataFrame(
    [
        {"strategy": name, **summarize(daily, cost_bps=COST_BPS)}
        for name, daily in daily_paths.items()
    ]
)
display(strategy_results)

,0
selected_position_cells,178
missing_selected_returns,0
affected_sessions,0


,strategy,n_sessions,active_sessions,mean_gross,mean_net,sharpe_gross,sharpe_net,total_return_gross,total_return_net,max_drawdown,mean_turnover,total_turnover,breakeven_bps_per_side,cost_bps_per_side
0,all_headlines,167,40,-0.000763,-0.001172,-1.736735,-2.635057,-0.123223,-0.181313,-0.203492,0.409496,68.385812,-18.624863,10.000000
1,first_release_only,167,43,-0.000627,-0.001079,-1.473352,-2.502392,-0.102964,-0.168192,-0.190247,0.451167,75.344855,-13.908328,10.000000


## Accounting validation, costs, paired inference, and temporal halves

In [5]:
def fold_final_liquidation(frame: pd.DataFrame) -> pd.DataFrame:
    liquidation = frame.loc[frame["final_liquidation"]]
    intervals = frame.loc[~frame["final_liquidation"]].copy().reset_index(drop=True)
    if len(liquidation) != 1 or intervals.empty:
        raise RuntimeError("expected exactly one terminal liquidation")
    final = liquidation.iloc[0]
    last = intervals.index[-1]
    intervals.loc[last, "net_return"] = (
        (1.0 + intervals.loc[last, "net_return"]) * (1.0 + final["net_return"])
        - 1.0
    )
    intervals.loc[last, "turnover"] += final["turnover"]
    intervals.loc[last, "cost"] += final["cost"]
    return intervals


def matrix_to_targets(weights: np.ndarray) -> tuple[TargetPortfolio, ...]:
    targets = []
    for session, vector in zip(sessions, weights, strict=True):
        nonzero = np.flatnonzero(~np.isclose(vector, 0.0))
        positions = tuple(
            PositionTarget(
                symbol=SYMBOLS[index],
                action=float(np.sign(vector[index])),
                volatility=None,
                raw_weight=float(vector[index]),
                target_weight=float(vector[index]),
                exclusion_reason=None,
            )
            for index in nonzero
        )
        long_exposure = float(vector[vector > 0].sum())
        short_exposure = float(-vector[vector < 0].sum())
        targets.append(
            TargetPortfolio(
                session=str(pd.Timestamp(session).date()),
                positions=positions,
                gross_exposure=long_exposure + short_exposure,
                net_exposure=long_exposure - short_exposure,
                long_exposure=long_exposure,
                short_exposure=short_exposure,
                cash_weight=1.0 - (long_exposure - short_exposure),
            )
        )
    return tuple(targets)


def matrix_to_returns(weights: np.ndarray) -> list[OpenToOpenReturn]:
    rows = []
    for session, next_session, vector, asset_returns in zip(
        sessions, next_sessions, weights, safe_returns, strict=True
    ):
        rows.append(
            OpenToOpenReturn(
                symbol="__CLOCK__",
                session=str(pd.Timestamp(session).date()),
                next_session=str(pd.Timestamp(next_session).date()),
                value=0.0,
            )
        )
        for index in np.flatnonzero(~np.isclose(vector, 0.0)):
            rows.append(
                OpenToOpenReturn(
                    symbol=SYMBOLS[index],
                    session=str(pd.Timestamp(session).date()),
                    next_session=str(pd.Timestamp(next_session).date()),
                    value=float(asset_returns[index]),
                )
            )
    return rows


accounting_rows = []
for name, weights in (
    ("all_headlines", baseline_weights),
    ("first_release_only", filtered_weights),
):
    ledger = ledger_rows_to_frame(
        run_open_to_open_ledger(
            matrix_to_targets(weights),
            matrix_to_returns(weights),
            cost_rate_per_side=COST_BPS / 10_000.0,
            initial_nav_usd=INITIAL_NAV,
            force_final_liquidation=True,
        )
    )
    ledger = fold_final_liquidation(ledger)
    errors = {
        column: float((ledger[column] - daily_paths[name][column]).abs().max())
        for column in ("gross_return", "net_return", "turnover")
    }
    if max(errors.values()) > 1e-12:
        raise RuntimeError(f"{name} matrix accounting disagrees with ledger: {errors}")
    accounting_rows.append({"strategy": name, **errors})
accounting_audit = pd.DataFrame(accounting_rows)

cost_rows = []
for cost_bps in COST_GRID:
    for name, weights in (
        ("all_headlines", baseline_weights),
        ("first_release_only", filtered_weights),
    ):
        daily = matrix_backtest(weights, safe_returns, cost_bps=cost_bps)
        cost_rows.append(
            {"strategy": name, **summarize(daily, cost_bps=cost_bps)}
        )
cost_curve = pd.DataFrame(cost_rows)

cash = daily_paths["first_release_only"][["session_date"]].copy()
cash["net_return"] = 0.0
comparison_specs = (
    ("first_release_minus_all_headlines", "first_release_only", "all_headlines"),
    ("first_release_minus_cash", "first_release_only", "cash"),
)
comparison_frames = {**daily_paths, "cash": cash}
inference_rows = []
for index, (comparison, challenger, baseline) in enumerate(comparison_specs):
    inference_rows.append(
        {
            "comparison": comparison,
            **paired_block_bootstrap_difference(
                comparison_frames[challenger],
                comparison_frames[baseline],
                value_col="net_return",
                block_length=BLOCK_LENGTH,
                replications=BOOTSTRAP_REPLICATIONS,
                seed=SEED + index,
            ),
        }
    )
inference = pd.DataFrame(inference_rows)
inference["q_bh"] = benjamini_hochberg(inference["p_two_sided"].tolist())
inference["multiplicity_family"] = (
    "two frozen story-family comparisons; BH-FDR q=0.05"
)

half_rows = []
midpoint = len(sessions) // 2
for period, indices in (
    ("first_half", np.arange(midpoint)),
    ("second_half", np.arange(midpoint, len(sessions))),
):
    for name, daily in daily_paths.items():
        half_rows.append(
            {
                "period": period,
                "strategy": name,
                **summarize(daily.iloc[indices].reset_index(drop=True), cost_bps=COST_BPS),
            }
        )
evaluation_halves = pd.DataFrame(half_rows)
display(accounting_audit)
display(inference)
display(evaluation_halves)

,strategy,gross_return,net_return,turnover
0,all_headlines,0.000000,0.000000,0.000000
1,first_release_only,0.000000,0.000000,0.000000


,comparison,n_sessions,mean_difference,ci_low,ci_high,p_two_sided,block_length,replications,seed,q_bh,multiplicity_family
0,first_release_minus_all_headlines,167,0.000094,-0.000065,0.000262,0.262100,5,9999,20260816,0.262100,two frozen story-family comparisons; BH-FDR q=...
1,first_release_minus_cash,167,-0.001079,-0.002011,-0.000208,0.018300,5,9999,20260817,0.036600,two frozen story-family comparisons; BH-FDR q=...


,period,strategy,n_sessions,active_sessions,mean_gross,mean_net,sharpe_gross,sharpe_net,total_return_gross,total_return_net,max_drawdown,mean_turnover,total_turnover,breakeven_bps_per_side,cost_bps_per_side
0,first_half,all_headlines,83,16,-0.000313,-0.000598,-0.854839,-1.613856,-0.027052,-0.049777,-0.075519,0.284231,23.591201,-11.027134,10.000000
1,first_half,first_release_only,83,17,-0.000357,-0.000671,-0.977557,-1.818099,-0.030533,-0.055527,-0.080569,0.314126,26.072440,-11.356768,10.000000
2,second_half,all_headlines,84,24,-0.001207,-0.001740,-2.406950,-3.427584,-0.098844,-0.138427,-0.138427,0.533269,44.794610,-22.626227,10.000000
3,second_half,first_release_only,84,26,-0.000895,-0.001481,-1.863724,-3.051253,-0.074712,-0.119289,-0.119289,0.586576,49.272415,-15.258483,10.000000


## Frozen gates and figures

In [6]:
result_lookup = strategy_results.set_index("strategy")
inference_lookup = inference.set_index("comparison")
filtered_result = result_lookup.loc["first_release_only"]
improvement_test = inference_lookup.loc["first_release_minus_all_headlines"]
cash_test = inference_lookup.loc["first_release_minus_cash"]
filtered_half_sharpes = evaluation_halves.loc[
    evaluation_halves["strategy"].eq("first_release_only"), "sharpe_net"
]
improvement_gate = bool(
    improvement_test["mean_difference"] > 0
    and improvement_test["ci_low"] > 0
    and improvement_test["q_bh"] < 0.05
)
standalone_gate = bool(
    filtered_result["sharpe_net"] > 0
    and filtered_result["breakeven_bps_per_side"] > COST_BPS
    and cash_test["ci_low"] > 0
    and cash_test["q_bh"] < 0.05
    and filtered_half_sharpes.ge(0).all()
)
gates = {
    "return_free_feasibility_gate": input_gate,
    "improvement_gate": improvement_gate,
    "standalone_quality_gate": standalone_gate,
    "retrospective_promotion_allowed": False,
}
display(pd.Series(gates, name="pass").to_frame())

fig, axes = plt.subplots(2, 1, figsize=(10.8, 7.0), sharex=True)
for label, colour, suffix in (
    ("All headlines", CATEGORICAL[0], "baseline"),
    ("First release only", CATEGORICAL[1], "first_release"),
):
    axes[0].plot(
        schedule_pair["session_date"],
        schedule_pair[f"dispersion_{suffix}"],
        color=colour,
        label=label,
        alpha=0.9,
    )
    active_rows = schedule_pair.loc[schedule_pair[f"active_{suffix}"]]
    axes[0].scatter(
        active_rows["session_date"],
        active_rows[f"dispersion_{suffix}"],
        color=colour,
        s=18,
        zorder=3,
    )
axes[0].set_ylabel("q90−q10 mean score")
axes[0].set_title("Story-family filtering changes the frozen event schedule")
axes[0].legend(loc="upper left")

axes[1].plot(
    daily_rank_spearman.index,
    daily_rank_spearman.to_numpy(),
    color=CATEGORICAL[2],
)
axes[1].axhline(1.0, color=INK["reference"], linestyle="--", linewidth=1)
axes[1].set_ylabel("Daily rank Spearman")
axes[1].set_xlabel("Decision session")
axes[1].set_title("Cross-sectional ranks remain close after revision removal")
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "story_family_input_and_schedule.png",
    dpi=170,
    bbox_inches="tight",
)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.1))
for name, label, colour in (
    ("all_headlines", "All headlines", CATEGORICAL[0]),
    ("first_release_only", "First release only", CATEGORICAL[1]),
):
    daily = daily_paths[name]
    axes[0].plot(
        daily["session_date"],
        np.cumprod(1.0 + daily["net_return"]),
        label=label,
        color=colour,
    )
axes[0].axhline(1.0, color=INK["reference"], linestyle="--", linewidth=1)
axes[0].set_title("Net path at 10 bps/side")
axes[0].set_ylabel("Growth of 1")
axes[0].tick_params(axis="x", rotation=35)
axes[0].legend(loc="best")

for name, label, colour in (
    ("all_headlines", "All headlines", CATEGORICAL[0]),
    ("first_release_only", "First release only", CATEGORICAL[1]),
):
    use = cost_curve.loc[cost_curve["strategy"].eq(name)]
    axes[1].plot(
        use["cost_bps_per_side"],
        use["sharpe_net"],
        marker="o",
        label=label,
        color=colour,
    )
axes[1].axhline(0.0, color=INK["reference"], linestyle="--", linewidth=1)
axes[1].set_title("Cost sensitivity")
axes[1].set_xlabel("Cost per side (bps)")
axes[1].set_ylabel("Net Sharpe")

half_plot = evaluation_halves.pivot(
    index="period", columns="strategy", values="sharpe_net"
).reindex(["first_half", "second_half"])
x = np.arange(len(half_plot))
width = 0.36
axes[2].bar(
    x - width / 2,
    half_plot["all_headlines"],
    width,
    label="All headlines",
    color=CATEGORICAL[0],
)
axes[2].bar(
    x + width / 2,
    half_plot["first_release_only"],
    width,
    label="First release only",
    color=CATEGORICAL[1],
)
axes[2].axhline(0.0, color=INK["reference"], linestyle="--", linewidth=1)
axes[2].set_xticks(x, ["First half", "Second half"])
axes[2].set_title("Temporal stability")
axes[2].set_ylabel("Net Sharpe")
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "performance_and_costs.png",
    dpi=170,
    bbox_inches="tight",
)
plt.show()

,pass
return_free_feasibility_gate,True
improvement_gate,False
standalone_quality_gate,False
retrospective_promotion_allowed,False


findfont: Failed to find font weight medium, now using 400.


/var/folders/dg/_nj_7w2d4mlc56b3vr3vb51m0000gn/T/ipykernel_9868/1963956099.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/dg/_nj_7w2d4mlc56b3vr3vb51m0000gn/T/ipykernel_9868/1963956099.py:134: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Persist aggregate evidence and decision

In [7]:
for name, daily in daily_paths.items():
    daily.to_parquet(OUTPUT_DIR / f"{name}_daily.parquet", index=False)
baseline_schedule.to_csv(OUTPUT_DIR / "all_headlines_schedule.csv", index=False)
filtered_schedule.to_csv(OUTPUT_DIR / "first_release_schedule.csv", index=False)
input_comparison.to_csv(OUTPUT_DIR / "input_comparison.csv", index=False)
outcome_audit.to_csv(OUTPUT_DIR / "outcome_availability_audit.csv", index=False)
strategy_results.to_csv(OUTPUT_DIR / "strategy_results.csv", index=False)
cost_curve.to_csv(OUTPUT_DIR / "cost_curve.csv", index=False)
inference.to_csv(OUTPUT_DIR / "inference.csv", index=False)
evaluation_halves.to_csv(OUTPUT_DIR / "evaluation_halves.csv", index=False)
accounting_audit.to_csv(OUTPUT_DIR / "accounting_audit.csv", index=False)

status = (
    "RETROSPECTIVE_STORY_FAMILY_PASS_NOT_CONFIRMATION"
    if improvement_gate and standalone_gate
    else "RETROSPECTIVE_STORY_FAMILY_NO_GO"
)
manifest = {
    "status": status,
    "notebook": "43_lseg_gemma_story_family_first_release.ipynb",
    "git_commit_at_execution": subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip(),
    "seed": SEED,
    "input": {
        "notebook39_manifest": str(NOTEBOOK39_MANIFEST.relative_to(REPO_ROOT)),
        "notebook39_manifest_sha256": sha256_file(NOTEBOOK39_MANIFEST),
        "gemma_score_path": str(GEMMA_PATH.relative_to(REPO_ROOT)),
        "gemma_score_sha256": score_audit["output_sha256"],
        "merged_corpus_path": str(MERGED_HEADLINES.relative_to(REPO_ROOT)),
        "merged_corpus_sha256": family_audit["corpus_sha256"],
        "licensed_headline_text_emitted": False,
        "return_loaded_only_after_input_gate": True,
    },
    "family_filter": family_audit,
    "frozen_specification": {
        "rule_origin": "Notebook 39 complete Gemma 26B mean-continuous construction",
        "universe": "original 33 LSEG companies",
        "story_filter": "earliest timestamped release per terminal-suffix story family",
        "signal": "mean continuous Gemma signed probability score",
        "event_intensity": "firm-open q90 minus q10",
        "event_gate": "current spread >= strictly-prior trailing-60 q75 after 40 observations",
        "long_names": TOP_BOTTOM_NAMES,
        "short_names": TOP_BOTTOM_NAMES,
        "single_name_cap": SINGLE_NAME_CAP,
        "gross_exposure": 1.0,
        "holding_intervals": 1,
        "cost_bps_per_side": COST_BPS,
        "accounting": "validated drift-aware open-to-open ledger plus terminal liquidation",
    },
    "input_feasibility_gate": {
        "minimum_removed_hashes": MIN_REMOVED_HASHES,
        "required_complete_sessions": 167,
        "minimum_active_dates": MIN_ACTIVE_DATES,
        "minimum_changed_decision_dates": 1,
        "passed": input_gate,
    },
    "input_comparison": input_comparison.iloc[0].to_dict(),
    "outcome_availability_audit": outcome_audit.iloc[0].to_dict(),
    "strategy_results": strategy_results.to_dict(orient="records"),
    "cost_curve": cost_curve.to_dict(orient="records"),
    "inference": inference.to_dict(orient="records"),
    "evaluation_halves": evaluation_halves.to_dict(orient="records"),
    "accounting_audit": accounting_audit.to_dict(orient="records"),
    "gates": gates,
    "limitations": [
        "the LSEG return window was already opened and this comparison is retrospective",
        "only 167 decision sessions are available",
        "story-family syntax is vendor-specific and may not transfer to other feeds",
        "the first release may omit economically material facts added in later revisions",
        "prices are split adjusted but not dividend adjusted",
        "borrow fees, variable spreads, impact, and intraday microstructure are omitted",
        "no result is promoted before Gate F1 or genuinely new-date validation",
    ],
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, default=str), encoding="utf-8"
)
print(json.dumps(manifest, indent=2, default=str))

baseline_result = result_lookup.loc["all_headlines"]
decision = f"""
## Decision

**{status.replace('_', ' ')}.** The return-free filter removes
**{family_audit['removed_later_revision_only_hashes']:,}** of
{len(gemma_scores):,} successful hashes ({family_audit['removed_share']:.2%}),
changes **{int(input_comparison.iloc[0]['changed_firm_opens']):,}** firm-opens and
**{int(input_comparison.iloc[0]['changed_decision_dates'])}** frozen decision
dates. Active-date Jaccard is {input_comparison.iloc[0]['active_date_jaccard']:.3f}.

At 10 bps/side, all-headline versus first-release-only gross Sharpe is
**{baseline_result['sharpe_gross']:.3f} / {filtered_result['sharpe_gross']:.3f}**
and net Sharpe is **{baseline_result['sharpe_net']:.3f} /
{filtered_result['sharpe_net']:.3f}**. Net return is
**{baseline_result['total_return_net']:.2%} / {filtered_result['total_return_net']:.2%}**;
the filtered break-even is **{filtered_result['breakeven_bps_per_side']:.2f}
bps/side**. The paired filtered-minus-baseline effect is
**{improvement_test['mean_difference'] * 10_000:+.3f} bps/session** with 95%
interval **[{improvement_test['ci_low'] * 10_000:+.3f},
{improvement_test['ci_high'] * 10_000:+.3f}]** and BH q =
**{improvement_test['q_bh']:.4f}**. The improvement gate is
**{'PASS' if improvement_gate else 'FAIL'}** and standalone-quality gate is
**{'PASS' if standalone_gate else 'FAIL'}**.

This is a bounded answer about revision handling, not fresh alpha evidence.
Do not choose a different family suffix, revision policy, threshold, breadth,
or horizon from these outcomes. Any retained rule still requires genuinely new
LSEG/Gemma dates.
"""
display(Markdown(decision))

{
  "status": "RETROSPECTIVE_STORY_FAMILY_NO_GO",
  "notebook": "43_lseg_gemma_story_family_first_release.ipynb",
  "git_commit_at_execution": "8736eebdbb254c8bc6416a09a352f6b5e2f6b1b3",
  "seed": 20260816,
  "input": {
    "notebook39_manifest": "final_experiments/outputs/39_gemma_continuous_portability/manifest.json",
    "notebook39_manifest_sha256": "f24bff1a8ef71c1f7296c0bee3eba5a3ec4c2adde1bf7b8135deb05b80cd73e1",
    "gemma_score_path": "Data/collections/lseg_us_sector_44_8m_headlines/derived/headline_scores_gemma4_26b_a4b_it_deepinfra_fp8_investor_headline_soft_label_v1.csv",
    "gemma_score_sha256": "afb2d9c3029c4fba4fce3600218233a02d14051788f87313fd49c9da681c6d2f",
    "merged_corpus_path": "Data/collections/lseg_us_sector_44_8m_headlines/derived/merged/headlines.jsonl",
    "merged_corpus_sha256": "7f48c46690f3293cbfe846ddecc701852daf3c35643f2747dcbc39f829927878",
    "licensed_headline_text_emitted": false,
    "return_loaded_only_after_input_gate": true
  },
  "family_fil


## Decision

**RETROSPECTIVE STORY FAMILY NO GO.** The return-free filter removes
**8,314** of
888,155 successful hashes (0.94%),
changes **1,938** firm-opens and
**8** frozen decision
dates. Active-date Jaccard is 0.930.

At 10 bps/side, all-headline versus first-release-only gross Sharpe is
**-1.737 / -1.473**
and net Sharpe is **-2.635 /
-2.502**. Net return is
**-18.13% / -16.82%**;
the filtered break-even is **-13.91
bps/side**. The paired filtered-minus-baseline effect is
**+0.935 bps/session** with 95%
interval **[-0.652,
+2.619]** and BH q =
**0.2621**. The improvement gate is
**FAIL** and standalone-quality gate is
**FAIL**.

This is a bounded answer about revision handling, not fresh alpha evidence.
Do not choose a different family suffix, revision policy, threshold, breadth,
or horizon from these outcomes. Any retained rule still requires genuinely new
LSEG/Gemma dates.
